<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Save and Load Experiment Topologies

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to save experiment topologies to a local file and reload them later. If you find yourself repeatedly creating the same topology, or want to share a topology with collaborators, the save/load feature ensures consistency and saves time.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create a slice topology and **save** it to a GraphML file
2. **Load** a previously saved topology into a new slice
3. Submit the loaded topology to FABRIC for provisioning
4. Understand when and why to use topology files

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must** complete the environment setup:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook to create your `fabric_rc` and `ssh_config` files
2. Have a valid FABRIC account with an active project

**Tip:** This notebook creates and submits a slice. Make sure you delete it when finished to free resources for other users.

</div>

## Background: What is a Topology File?

A **topology file** is a [GraphML](http://graphml.graphite.org/) XML file that describes the structure of your experiment -- nodes, networks, interfaces, and their properties. When you call `slice.save()`, FABlib serializes the current topology request (not a running slice) to this file.


<div class="fab-danger">

**Important:** The save/load feature only stores the **topology request** (the blueprint). It does **not** save a running slice's state, IP addresses, or SSH keys. Loading a topology creates a brand-new slice request that must be submitted to FABRIC.

</div>

---

## Step 1: Import FABlib and Verify Configuration

Every FABRIC notebook starts by importing the FABlib library and verifying your configuration.

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Build a Topology and Save It

First, we create a slice topology (without submitting it) and save it to a `.graphml` file. This file can be inspected, shared, or version-controlled.

After running the cell below, you can view the saved file [here](./hello_fabric.graphml).

In [ ]:
# Create a new slice topology (not yet submitted to FABRIC)
saved_topology = fablib.new_slice(name="MySlice_Saved")

# Add a single node with default settings (random site, 2 cores, 8 GB RAM)
saved_topology.add_node(name="Node1")

# Save the topology request to a GraphML file
# This file contains the XML representation of your topology blueprint
saved_topology.save('hello_fabric.graphml')

<div class="fab-success">

**What just happened?** The topology (one node with default settings) was serialized to `hello_fabric.graphml` in the current directory. No resources were reserved on FABRIC -- we only saved the blueprint.

</div>

## Step 3: Load the Topology and Submit

Now we create a **new** slice and load the saved topology into it. The original `hello_fabric.graphml` file is unchanged by the load operation. After loading, we submit the slice to FABRIC for provisioning.

In [ ]:
# Create a new empty slice with a different name
loaded_topology = fablib.new_slice(name="MySlice_Loaded")

# Load the previously saved topology into this slice
# The node definitions and network configuration are read from the file
loaded_topology.load('hello_fabric.graphml')

# Submit the loaded topology to FABRIC for provisioning (~2-5 minutes)
loaded_topology.submit()

## Step 4: Inspect the Slice

Once the slice is active, verify that the loaded topology was provisioned correctly.

In [ ]:
# Show slice-level information (state, expiration, project, etc.)
loaded_topology.show()

# List all nodes in the slice with their details
loaded_topology.list_nodes();

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
loaded_topology.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `FileNotFoundError` on `load()` | GraphML file path is wrong or file was not saved | Verify the file exists with `!ls hello_fabric.graphml` |
| `show_config()` shows missing values | Environment not configured | Run [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) |
| Loaded slice has wrong topology | Loaded an outdated `.graphml` file | Re-save the topology and reload |
| `submit()` fails after `load()` | Resources no longer available at the site encoded in the file | Edit the topology or let FABRIC pick a random site |
| Saved file is empty or corrupted | `save()` was called before adding nodes | Add nodes/networks before calling `save()` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.save(filename)` | Save the topology request to a GraphML file | [save](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.save) |
| `slice.load(filename)` | Load a topology from a GraphML file | [load](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.load) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.show()` | Display slice attributes | [show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show) |
| `slice.list_nodes()` | List all nodes in the slice | [list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodes) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you know how to save and load topologies, explore these notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Hello FABRIC** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first experiment from scratch |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Networking** | [FABnet IPv4 (auto)](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Build multi-node topologies with networking |
| **Advanced Scheduling** | [advanced_scheduling_slice](../create_slice/advanced_scheduling_slice.ipynb) | Reserve resources for a future time window |
| **Listing Resources** | [list_all_resources](../sites_and_resources/list_all_resources.ipynb) | Query available capacity across all sites |